# ML S1 · Notebook 00 — The Housing Dataset

**Course:** Machine Learning · Session 1 of 10
**Role of this notebook:** the single source of the dataset used in ML S1 **and** ML S2. Run it once and it writes `housing.csv` next to the notebook. Every other notebook this week imports that file — the data is never re-created by hand, so everyone in the room is working with *exactly* the same numbers.

> **You do not need to understand every line of the generator below.** This notebook exists so the data is available with nothing to download. Skim the code, then spend your attention on the **data dictionary** and the **"What to notice already"** section at the end — those two are what the rest of the week builds on.


## Why we *generate* data instead of downloading it

Three practical reasons:

1. **Nothing to download.** You run one cell, the data appears. No Kaggle account, no broken link, no "it worked on my laptop."
2. **Everyone gets identical data.** We fix a random *seed*, so all 25 of us produce the same 5,000 rows. When the notes say "row 4 is San Francisco," it is San Francisco on your screen too.
3. **We control the lesson.** Real datasets are messy in random ways. Here we can plant *specific* problems on purpose — a few impossible values, a few data-entry errors — so we can practise catching them.

**One honesty note.** This data is *synthetic* — invented by a formula, not collected from real sales. That makes it cleaner than reality. Real-world data is worse: messier, more surprising, harder to trust. We start clean so the ideas are visible, then meet real mess in your certification projects. Keep that caveat in mind: never present synthetic results as if they were real market findings.


In [1]:
# --- Setup ---
import numpy as np
import pandas as pd

# The seed is what makes everyone's data identical. Change it and you get a
# different (but statistically similar) dataset. Leave it as 42 to match the notes.
SEED = 42
rng = np.random.default_rng(SEED)

N = 5000  # number of houses


## The generator

Read this as a *story about what drives house prices*, not as code to memorise:

- Each house sits in one of **10 US metros**, grouped into three **price tiers** (Tier 1 = priciest cities like San Francisco and New York, Tier 3 = most affordable like Columbus and Indianapolis).
- Bigger `sqft`, more `bedrooms`/`bathrooms`, more `garage_spaces`, and a bigger `lot_size` push the price **up**.
- Older houses (`house_age`) push it gently **down**.
- The city multiplier matters most of all — the same house is worth far more in San Francisco than in Phoenix.
- Finally we add **random noise**, because real prices are never a perfect formula. This noise is deliberate: it is why no model will ever predict these prices perfectly, and why measuring error will actually mean something.


In [2]:
# Each metro maps to (price tier, price multiplier).
# Tier 1 = most expensive cities; Tier 3 = most affordable.
metros = {
    'San Francisco': (1, 1.90), 'New York': (1, 1.75), 'Boston': (1, 1.45),
    'Seattle': (2, 1.35), 'Austin': (2, 1.20), 'Denver': (2, 1.15), 'Chicago': (2, 1.05),
    'Phoenix': (3, 0.90), 'Columbus': (3, 0.80), 'Indianapolis': (3, 0.75),
}
metro_names = list(metros)

# How common each metro is in our data (must sum to 1).
metro_probs = np.array([.11, .13, .08, .10, .11, .09, .12, .10, .08, .08])
metro_probs = metro_probs / metro_probs.sum()

metro = rng.choice(metro_names, size=N, p=metro_probs)
location_tier = np.array([metros[m][0] for m in metro])
city_multiplier = np.array([metros[m][1] for m in metro])


In [3]:
# House characteristics
sqft      = rng.normal(1900, 650, N).clip(500, 6000).round(0)
bedrooms  = np.clip(np.round(sqft / 650 + rng.normal(0, 0.7, N)), 1, 7).astype(int)
bathrooms = np.clip(np.round(bedrooms * 0.6 + rng.normal(0, 0.4, N)), 1, 5).astype(int)
house_age = rng.integers(0, 80, N)
garage_spaces = rng.choice([0, 1, 2, 3], size=N, p=[.15, .45, .30, .10])
lot_size  = (sqft * rng.uniform(1.2, 4.0, N)).round(0)


In [4]:
# The price 'signal' — how each feature contributes, before noise.
base = 90_000
price = (
    base
    + sqft * 180 * city_multiplier   # size matters more in pricey cities
    + bedrooms * 8_000
    + bathrooms * 12_000
    + garage_spaces * 9_000
    + lot_size * 6
    - house_age * 1_200              # older = slightly cheaper
) * city_multiplier

# Deliberate 12% random noise: real prices are never a clean formula.
price = price * rng.normal(1.0, 0.12, N)
price = price.clip(60_000, None).round(-2)   # no negative/tiny prices; round to nearest $100


In [5]:
# Assemble the table
df = pd.DataFrame({
    'metro': metro,
    'location_tier': location_tier,
    'sqft': sqft.astype(int),
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'house_age': house_age,
    'garage_spaces': garage_spaces,
    'lot_size': lot_size.astype(int),
    'price': price.astype(int),
})

df.shape


(5000, 9)

## Now we deliberately dirty the data

Real datasets contain mistakes: someone fat-fingers a number, a form lets through an impossible value. We plant a handful of these **on purpose** so that in the build notebook you get to practise finding and fixing them — instead of assuming clean data and getting burned later.

We plant two kinds:
- **6 data-entry outliers:** `sqft` multiplied by 10 (a 1,800 sqft house recorded as 18,000).
- **6 impossible values:** `bedrooms = 0` (a house with no bedrooms).


In [6]:
bad = rng.choice(N, 12, replace=False)
df.loc[bad[:6],  'sqft']     = df.loc[bad[:6],  'sqft'] * 10   # data-entry outliers
df.loc[bad[6:],  'bedrooms'] = 0                                # impossible values

# Save for the other notebooks to import
df.to_csv('housing.csv', index=False)
print('Saved housing.csv with', len(df), 'rows and', df.shape[1], 'columns.')


Saved housing.csv with 5000 rows and 9 columns.


## Data dictionary

| Column | Meaning | Notes |
|---|---|---|
| `metro` | US city the house is in | 10 cities; the only text column |
| `location_tier` | City price tier: **1 = priciest**, 3 = most affordable | The single strongest signal in the data |
| `sqft` | Living area, square feet | Contains a few planted outliers |
| `bedrooms` | Number of bedrooms | Contains a few planted impossible `0` values |
| `bathrooms` | Number of bathrooms | Correlated with bedrooms and sqft |
| `house_age` | Years since built (0–79) | Weak, slightly negative effect on price |
| `garage_spaces` | Garage capacity, 0–3 | **Deliberately weak** — barely affects price |
| `lot_size` | Lot area, square feet | Moderate positive effect |
| **`price`** | **TARGET** — sale price in **USD** | What we will learn to predict |

**Two unit reminders that will bite you if you forget them** (this is a real professional habit — always read the dictionary before trusting a number):
- `price` is in **whole US dollars**. A value of `625000` means \$625,000.
- `location_tier` is *inverted* from what you might expect: a **higher** tier number means a **cheaper** city.


In [7]:
# A quick look
df.head(8)


,metro,location_tier,sqft,bedrooms,bathrooms,house_age,garage_spaces,lot_size,price
0,Phoenix,3,1186,3,1,37,1,2158,250900
1,Austin,2,1548,4,2,22,1,3074,497100
2,Columbus,3,2077,3,2,49,2,4723,429300
3,Chicago,2,1492,2,2,66,2,5467,376600
4,San Francisco,1,1719,2,2,25,2,6593,1466500
5,Indianapolis,3,2187,4,2,9,2,4993,293600
6,Phoenix,3,1673,1,1,5,3,2224,406900
7,Phoenix,3,2047,3,2,4,1,5901,392000


In [8]:
# Summary statistics — notice the extremes
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
metro,5000,10,New York,677,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_tier,5000.0,NaN,NaN,NaN,1.93,0.763424,1.0,1.0,2.0,3.0,3.0
sqft,5000.0,NaN,NaN,NaN,1919.538,836.133852,500.0,1469.0,1888.5,2337.0,24620.0
bedrooms,5000.0,NaN,NaN,NaN,2.9464,1.216151,0.0,2.0,3.0,4.0,7.0
bathrooms,5000.0,NaN,NaN,NaN,1.839,0.792215,1.0,1.0,2.0,2.0,5.0
house_age,5000.0,NaN,NaN,NaN,40.0796,23.169226,0.0,20.0,40.0,61.0,79.0
garage_spaces,5000.0,NaN,NaN,NaN,1.3502,0.847646,0.0,1.0,1.0,2.0,3.0
lot_size,5000.0,NaN,NaN,NaN,4944.5886,2342.904036,605.0,3180.75,4581.5,6435.75,15749.0
price,5000.0,NaN,NaN,NaN,760643.44,482519.589107,81600.0,393175.0,625650.0,1008400.0,2993900.0


## What to notice already

Run the two cells above and look, before we ever build a model:

1. **`sqft` has a maximum in the tens of thousands.** No normal house is 24,000 sqft — those are our planted data-entry errors. Clean-looking numeric data still hides nonsense.
2. **`bedrooms` has a minimum of `0`.** A house with zero bedrooms is impossible — another planted error.
3. **`price` ranges from about \$80k to nearly \$3M.** That huge spread is mostly the city multiplier. A model has to work across all of it.
4. **`garage_spaces` will turn out to barely matter.** Hold that thought — in ML S2 we will watch a technique called Lasso *discover* that on its own and drop it.

We will **not** fix these problems here. Finding and fixing them is the first real task in the build notebook (`ML_S1_02_build_and_ship`). Leaving them in is the point.

---

**Next:** open `ML_S1_01_foundations.ipynb`. You will not need this notebook again until the build — but if you ever want fresh data, just re-run this one.
